# Lekce 02 - Prozkoumání Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** je jednotný rámec pro tvorbu AI agentů. Poskytuje čistou, kompozitní architekturu se čtyřmi základními stavebními bloky:

- **Klient** – připojuje se k AI modelu a zajišťuje komunikaci
- **Agent** – obaluje klienta s instrukcemi a definicemi nástrojů
- **Nástroje** – rozšiřují schopnosti agenta o vlastní funkce, které může model volat
- **Seance** – uchovává historii konverzace pro vícekrokové interakce

V této lekci vytvoříme **agenta pro rezervaci cest**, který pomocí těchto konceptů ověří dostupnost cílových destinací.


## Nastavení


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pochopení architektury Agent Frameworku

Microsoft Agent Framework následuje vrstevnatou architekturu:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klient** – `AzureAIProjectAgentProvider` se připojuje k nasazení Azure OpenAI. Zajišťuje autentizaci, formátování požadavků a zpracování odpovědí.
2. **Agent** – Vytvořen z klienta pomocí `provider.create_agent()`, agent kombinuje přístup k modelu s instrukcemi (systémovým promptem) a nástroji.
3. **Nástroje** – Funkce v Pythonu dekorované `@tool`, které agent může vyvolat pro provedení akcí nebo získání dat.
4. **Relace** – Objekt `AgentSession` (vytvořený pomocí `agent.create_session()`), který uchovává historii konverzace, umožňující víceturnou dialog, kde si agent pamatuje předchozí kontext.

Pojďme vybudovat každou vrstvu krok za krokem.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Přidávání nástrojů pomocí dekorátoru @tool

Nástroje umožňují agentům provádět akce nad rámec generování textu. Dekorátor `@tool` převádí běžnou Python funkci na něco, co může agent volat.

Klíčové body:
- Používejte `Annotated[type, "popis"]`, aby model rozuměl každému parametru.
- Docstring se stává popisem nástroje, který model vidí.
- `approval_mode="never_require"` znamená, že nástroj běží automaticky bez potvrzení uživatele.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Vytváření agenta s nástroji

Nyní zkombinujeme klienta, instrukce a nástroje do agenta. `instructions` fungují jako systémový prompt — určují osobnost a chování agenta.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Vícekolové konverzace se relacemi

`AgentSession` (vytvořená pomocí `agent.create_session()`) sleduje všechny zprávy v konverzaci. Předáváním téže relace do každého volání `agent.run()` má agent přístup k celé historii konverzace a může se odkazovat na předchozí zprávy.

Předáváme `tools=[check_destination_availability]`, aby agent mohl během každého kola volat náš kontrolor dostupnosti.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Shrnutí

V této lekci jste prozkoumali čtyři pilíře Microsoft Agent Frameworku:

| Pojem | Co jste se naučili |
|---------|------------------|
| **Klient** | `AzureAIProjectAgentProvider` se připojuje k Azure OpenAI pomocí autentizace založené na přihlašovacích údajích |
| **Agent** | `provider.create_agent()` spojuje připojení modelu s instrukcemi a názvem |
| **Nástroje** | Dekorátor `@tool` zpřístupňuje Python funkce, které agent může volat |
| **Relace** | `agent.create_session()` udržuje historii konverzace přes více kol |

Tyto stavební bloky se skládají dohromady, aby vytvořily agenty, kteří mohou vést přirozené rozhovory, volat externí funkce a udržovat kontext — základ pro pokročilejší agentní vzory v následujících lekcích.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:  
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Původní dokument v jeho mateřském jazyce by měl být považován za závazný zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za žádné nedorozumění nebo mylné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
